# Runner-control action representation audit

This notebook executes the frozen v1.1 protocol on the deduplicated feasibility-audit and aggregate-benchmark feed union. It catalogs explicit action evidence, builds first-base-only exposure windows, and joins detailed actions to the frozen Savant benchmark. It does not estimate threat-tempo effects and does not change the pending video-validation gate.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.runner_control_audit import run_audit

ROOT

WindowsPath('C:/Users/isaac/CodingProjects/runner-threat-pitcher-tempo/runner-threat-pitcher-tempo')

## Execute the frozen audit

The call below regenerates every compact CSV in `results/`. Raw cached feeds remain ignored.

In [2]:
outputs = run_audit(ROOT, write=True)
outputs['summary']

,metric,value
0,source_manifest_rows,212
1,unique_games,211
2,duplicate_games,1
3,action_rows,1591
4,pickoff_throws,857
5,pitcher_pickoff_throws,801
6,catcher_pickoff_throws,55
7,step_offs,353
8,steal_attempts,319
9,opportunity_windows,12228


## Action and opportunity structure

In [3]:
action_breakdown = (
    outputs['actions']
    .groupby(['action_type', 'initiator_role', 'outcome_type'], dropna=False)
    .size().rename('rows').reset_index()
)
action_breakdown

,action_type,initiator_role,outcome_type,rows
0,balk,pitcher,advance,12
1,disengagement_violation,pitcher,advance,1
2,pickoff_throw,catcher,caught_stealing,1
3,pickoff_throw,catcher,error_advance,1
4,pickoff_throw,catcher,picked_off,3
5,pickoff_throw,catcher,safe_return,50
6,pickoff_throw,pitcher,caught_stealing,9
7,pickoff_throw,pitcher,error_advance,9
8,pickoff_throw,pitcher,picked_off,14
9,pickoff_throw,pitcher,safe_return,769


In [4]:
opportunity_breakdown = (
    outputs['opportunities']
    .groupby(['open_timing', 'close_reason', 'any_attempt'])
    .size().rename('windows').reset_index()
)
opportunity_breakdown

,open_timing,close_reason,any_attempt,windows
0,after_pitch,next_pitch,False,8144
1,after_pitch,next_pitch,True,514
2,after_pitch,plate_appearance_end,True,1
3,after_pitch,state_exit,False,79
4,after_pitch,state_exit,True,229
5,after_state_entry,next_pitch,False,2
6,before_first_event,next_pitch,False,3125
7,before_first_event,next_pitch,True,128
8,before_first_event,state_exit,True,6


In [5]:
outputs['conflict_sensitivity']

,scenario,opportunity_windows,windows_with_attempt,action_rows
0,primary_include_known_conflict,12228,878,1591
1,sensitivity_exclude_game_746690_pa_38,12228,878,1591


## Contribution to the aggregate tempo benchmark

Categories are nonexclusive. Exclusions remain labeled sensitivities; the documented-rule population is unchanged.

In [6]:
outputs['tempo_contributions']

,base_state,action_category,intervals_nonexclusive,median_start_to_start_seconds,median_end_to_end_seconds
0,bases_empty,mound_visit,8,50.3670,49.9770
1,bases_empty,other_nonpitch,3,41.9310,45.3200
2,bases_empty,pickoff_throw_pitcher,1,168.3590,169.5180
3,bases_empty,steal_attempt,4,37.4740,35.6630
4,bases_empty,timeout,173,24.9770,25.6950
5,bases_empty,timer_violation,2,38.2040,38.4285
6,runners_on,balk,1,44.3760,44.8550
7,runners_on,disengagement_violation,1,86.0640,85.8760
8,runners_on,mound_visit,15,59.4190,62.1270
9,runners_on,other_nonpitch,23,35.2040,34.0310


In [7]:
(outputs['tempo_sensitivity']
 .groupby('scenario', as_index=False)
 .agg(cells=('feed_median_seconds', 'size'),
      median_absolute_savant_gap=('feed_minus_savant_seconds', lambda x: x.abs().median()),
      maximum_absolute_savant_gap=('feed_minus_savant_seconds', lambda x: x.abs().max()),
      median_cell_shift=('median_change_seconds', lambda x: x.abs().median())))


,scenario,cells,median_absolute_savant_gap,maximum_absolute_savant_gap,median_cell_shift
0,documented_rules,12,0.1530,0.5240,0.00000
1,exclude_any_nonpitch_action,12,0.0010,0.0265,0.14100
2,exclude_balk,12,0.1530,0.5240,0.00000
3,exclude_disengagement_violation,12,0.1530,0.5240,0.00000
4,exclude_mound_visit,12,0.1515,0.4870,0.00200
5,exclude_other_nonpitch,12,0.1525,0.4960,0.00425
6,exclude_pickoff_throw_catcher,12,0.1530,0.4960,0.00000
7,exclude_pickoff_throw_pitcher,12,0.1390,0.3450,0.00825
8,exclude_steal_attempt,12,0.1525,0.4740,0.00100
9,exclude_step_off,12,0.1530,0.4805,0.00050


## Focused reproducibility checks

In [8]:
assert outputs['game_inventory']['game_pk'].nunique() == 211
assert not outputs['actions']['action_id'].duplicated().any()
assert not outputs['opportunities']['opportunity_id'].duplicated().any()
assert len(outputs['tempo_interval_join']) == 485
assert outputs['opportunities']['known_state_conflict'].sum() == 0
assert set(outputs['actions'].loc[outputs['actions']['event_record_type'].eq('pickoff'), 'initiator_role']) <= {'pitcher', 'catcher', 'unknown'}
assert outputs['review_queue']['sample_seed'].eq(20241117).all()
'All focused checks passed.'

'All focused checks passed.'